In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
sys.path.append(str(Path("..")))
from src.dataset import train_data, train_loader, val_loader, test_loader,test_path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader


torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([2, 1, 3, 5, 6, 2, 3, 5, 5, 3])


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [4]:
num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)

print(model)

CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=8, bias=True)
  )
)


In [5]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

In [6]:
num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []


In [7]:
for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 1.7302 | Train Acc: 0.3964 | Val Loss: 1.1818 | Val Acc: 0.5916 | Val Macro F1: 0.5432
F1_Score by each class
[0.58181818 0.72941176 0.50485437 0.12244898 0.62721893 0.67164179
 0.78362573 0.32432432]
Epoch [2/100] | Train Loss: 0.8305 | Train Acc: 0.7031 | Val Loss: 0.6866 | Val Acc: 0.7838 | Val Macro F1: 0.7670
F1_Score by each class
[0.73684211 0.86170213 0.68367347 0.68449198 0.76923077 0.85067873
 0.91099476 0.63829787]
Epoch [3/100] | Train Loss: 0.4729 | Train Acc: 0.8247 | Val Loss: 0.8374 | Val Acc: 0.6847 | Val Macro F1: 0.6755
F1_Score by each class
[0.83783784 0.8700565  0.54304636 0.31007752 0.66165414 0.77094972
 0.8988764  0.51162791]
Epoch [4/100] | Train Loss: 0.2890 | Train Acc: 0.9054 | Val Loss: 0.7124 | Val Acc: 0.7883 | Val Macro F1: 0.7514
F1_Score by each class
[0.78911565 0.87437186 0.65240642 0.6744186  0.78857143 0.85585586
 0.92631579 0.45      ]
Epoch [5/100] | Train Loss: 0.1400 | Train Acc: 0.9527 | Val Loss: 0.8226 | Val Acc:

In [8]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7934
Test Macro F1: 0.7901
Test F1 by class:
[0.82758621 0.89795918 0.65384615 0.63917526 0.85714286 0.82882883
 0.94117647 0.675     ]
